## Запуск Spark-сессии (конфигурация под MAC)

In [ ]:
import os
import time

from pyspark.sql import SparkSession

os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация Кластера ---
# Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 6
MEMORY_PER_EXECUTOR_MB = 4096 

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"Запуск в режиме: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4") # Для тестов меньше дефолтных 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Проверка конфигурации ---
print(f"Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Проверка количества экзекуторов (может занять пару секунд на старт)
time.sleep(3) 
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 4. Тест на распределение (Пример) ---
# Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
def print_executor_info(iterator):
    import os
    # Получаем ID экзекутора из переменных окружения процесса
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Создаем датафрейм и применяем трансформацию
df = sp_s.range(0, 10, 1, 4) # 4 партиции
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

sp_s

Запуск в режиме: local-cluster[2, 6, 4096]


26/06/20 20:27:45 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 10.246.5.114 instead (on interface en0)
26/06/20 20:27:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 20:27:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Сессия создана.
Driver Memory Config: 4g
Executor Memory Config: 4g


📊 Активных экзекуторов (проверка через RDD): 2

🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 3088
Executor ID: Driver/Local, PID: 3087
Executor ID: Driver/Local, PID: 3091
Executor ID: Driver/Local, PID: 3092


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 56972)
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/Users/danilsamsutdinov/HypEx/.venv/lib/pytho

## Импорты (HypEx)

In [2]:
import csv
import random

import pandas as pd
import pyspark.pandas as ps

import warnings
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning
warnings.filterwarnings("ignore", category=PandasAPIOnSparkAdviceWarning)
from hypex import AATest, ABTest
from hypex.analyzers.aa import AAScoreAnalyzer, OneAAStatAnalyzer
from hypex.comparators import StatsTTest, StatsChi2Test, GroupSizes, GroupTTest, GroupKSTest, GroupChi2Test, GroupDifference
from hypex.comparators.abstract import Comparator
from hypex.comparators.stats_hypothesis_testing import StatsTTest
from hypex.dataset import ExperimentData, AdditionalTreatmentRole
from hypex.dataset import ExperimentData, AdditionalTreatmentRole, TargetRole
from hypex.dataset import (Dataset, 
                           SmallDataset, 
                           InfoRole, 
                           TargetRole, 
                           TreatmentRole, 
                           DatasetAdapter, 
                           DefaultRole, 
                           StratificationRole)
from hypex.dataset.roles import TargetRole, FeatureRole, InfoRole, StatisticRole
from hypex.experiments.base import Experiment, OnRoleExperiment
from hypex.experiments.base_complex import ParamsExperiment
from hypex.reporters import DatasetReporter
from hypex.reporters.aa import OneAADictReporter
from hypex.splitters import AASplitter
from hypex.utils import BackendsEnum, SpaceEnum
from hypex.utils import SpaceEnum
from hypex.ui.aa import AAOutput
from hypex.ui.base import ExperimentShell

from hypex.experiments import IfParamsExperiment

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


## Генерация тестовых данных

In [3]:
def generate_test_data(
    n_rows: int = 100,
    filename: str | None = None,
    return_df: bool = True
) -> pd.DataFrame | None:
    
    headers = ['user_id', 
               'signup_month', 
               'treat', 'pre_spends', 
               'post_spends', 'age', 'gender', 'industry']
    
    data = []
    current_id = 0
    
    while len(data) < n_rows:
        if current_id > 0 and current_id % 10 == 0:
            current_id += 1
            continue
            
        user_id = float(current_id)
        signup_month = float(random.randint(0, 11))
        treat = float(random.choice([0, 1]))
        
        pre_spends = random.uniform(450, 550)
        
        if treat == 0.0:
            post_spends = pre_spends * random.uniform(0.80, 0.90)
        else:
            post_spends = pre_spends * random.uniform(1.00, 1.10)
            
        age = float(random.randint(18, 70))
        gender = random.choice(['M', 'F'])
        industry = random.choice(['Logistics', 'E-commerce'])
        
        data.append([
            user_id, signup_month, treat, 
            round(pre_spends, 1), round(post_spends, 1), 
            age, gender, industry
        ])
        
        current_id += 1

    df = pd.DataFrame(data, columns=headers)
    
    if filename:
        df.to_csv(filename, index=False, encoding='utf-8')
    
    if return_df:
        return df

In [5]:
def create_ds(df: pd.DataFrame, 
              backend: BackendsEnum=BackendsEnum.spark) -> Dataset:
    return Dataset(
        roles={
            "user_id": InfoRole(float),
            "treat": TreatmentRole(),
            "pre_spends": TargetRole(),
            "post_spends": DefaultRole(),
            "gender": StratificationRole(str),
            "industry" : DefaultRole()
        }, 
        data=df, 
        session=sp_s,
        backend=backend
    )

## Запуск AA-теста

In [6]:
df = generate_test_data(n_rows=1000)

### pandas backend

In [7]:
ds = create_ds(df=df, backend=BackendsEnum.pandas)
exp_data = ExperimentData(ds)
aa_test = AATest(n_iterations=3)
result = aa_test.execute(exp_data)
exp_data.analysis_tables

100%|██████████| 3/3 [00:00<00:00, 34.64it/s]


{'ParamsExperiment┴┴AATest':       feature    group  difference %  difference  test mean  control mean GroupTTest pass  GroupTTest p-value GroupKSTest pass  GroupKSTest p-value
 0  pre_spends  control      0.086770      0.4344   501.0710      500.6366          NOT OK            0.810761           NOT OK             0.770437
 1        mean      all           NaN         NaN        NaN           NaN          NOT OK            0.810761           NOT OK             0.770437
 2  pre_spends   test_0     -0.232851     -1.1676   500.2700      501.4376          NOT OK            0.519804           NOT OK             0.369905
 3        mean      all           NaN         NaN        NaN           NaN          NOT OK            0.519804           NOT OK             0.369905
 4  pre_spends  control     -0.081667     -0.4092   500.6492      501.0584          NOT OK            0.821549           NOT OK             0.989680
 
 6 rows × 10 columns,
 'AAScoreAnalyzer┴┴aa score':    GroupTTest score  Gro

### spark backend

In [ ]:
ds = create_ds(df=df, backend=BackendsEnum.spark)
exp_data = ExperimentData(ds)
aa_test = AATest(n_iterations=3)
result = aa_test.execute(exp_data)
exp_data.analysis_tables

26/06/20 20:32:31 WARN AttachDistributedSequenceExec: clean up cached RDD(20) in AttachDistributedSequenceExec(86)
26/06/20 20:32:32 WARN AttachDistributedSequenceExec: clean up cached RDD(34) in AttachDistributedSequenceExec(314)
26/06/20 20:32:33 WARN AttachDistributedSequenceExec: clean up cached RDD(61) in AttachDistributedSequenceExec(795)
26/06/20 20:32:33 WARN AttachDistributedSequenceExec: clean up cached RDD(84) in AttachDistributedSequenceExec(1170)
26/06/20 20:32:33 WARN AttachDistributedSequenceExec: clean up cached RDD(111) in AttachDistributedSequenceExec(1423)
26/06/20 20:32:34 WARN AttachDistributedSequenceExec: clean up cached RDD(145) in AttachDistributedSequenceExec(1930)
26/06/20 20:32:34 WARN AttachDistributedSequenceExec: clean up cached RDD(168) in AttachDistributedSequenceExec(2273)
26/06/20 20:32:34 WARN AttachDistributedSequenceExec: clean up cached RDD(197) in AttachDistributedSequenceExec(2535)
26/06/20 20:32:34 WARN AttachDistributedSequenceExec: clean up c

{'ParamsExperiment┴┴AATest':       feature   group  difference %  difference  test mean  control mean GroupKSTest pass  GroupKSTest p-value StatsTTest pass  StatsTTest p-value
 0  pre_spends  test_0      0.149777      0.7496   501.2286      500.4790           NOT OK             0.498083          NOT OK            0.679462
 1        mean     all           NaN         NaN        NaN           NaN           NOT OK             0.498083          NOT OK            0.679462
 2  pre_spends  test_0     -0.451406     -2.2660   499.7208      501.9868           NOT OK             0.359903          NOT OK            0.211480
 3        mean     all           NaN         NaN        NaN           NaN           NOT OK             0.359903          NOT OK            0.211480
 4  pre_spends  test_0      0.400999      2.0044   501.8560      499.8516           NOT OK             0.359903          NOT OK            0.269089
 
 6 rows × 10 columns,
 'AAScoreAnalyzer┴┴aa score':    GroupKSTest score  GroupKST

26/06/20 23:12:43 WARN TransportChannelHandler: Exception in connection from /10.246.5.114:56940
java.io.IOException: Can't assign requested address
	at java.base/sun.nio.ch.FileDispatcherImpl.read0(Native Method)
	at java.base/sun.nio.ch.SocketDispatcher.read(SocketDispatcher.java:39)
	at java.base/sun.nio.ch.IOUtil.readIntoNativeBuffer(IOUtil.java:276)
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:233)
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:223)
	at java.base/sun.nio.ch.SocketChannelImpl.read(SocketChannelImpl.java:356)
	at io.netty.buffer.PooledByteBuf.setBytes(PooledByteBuf.java:254)
	at io.netty.buffer.AbstractByteBuf.writeBytes(AbstractByteBuf.java:1132)
	at io.netty.channel.socket.nio.NioSocketChannel.doReadBytes(NioSocketChannel.java:357)
	at io.netty.channel.nio.AbstractNioByteChannel$NioByteUnsafe.read(AbstractNioByteChannel.java:151)
	at io.netty.channel.nio.NioEventLoop.processSelectedKey(NioEventLoop.java:788)
	at io.netty.channel.nio.NioEventLoop.processSelec